# IOAI — 2025 Stage 2 Abnormal Distribution (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.pkl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-abnormal-distribution/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 비정상 분포 — 다중과제 잡음처리 (Abnormal Distribution, 베이스라인)

폴란드 AI 올림피아드 II · 2025 · 2단계. 28×28 흑백 이미지에 **가우시안(라벨0)** 또는 **균등(라벨1)** 잡음이
무작위 파라미터로 더해졌다. **하나의** 신경망으로 세 가지를 동시에 수행한다:
1. **잡음 제거**(denoising) → 원본 복원 (PSNR),
2. **잡음 종류 분류**(가우시안0/균등1) → accuracy,
3. **가우시안 파라미터 추정**(라벨0 한정) → μ·σ 추정 (MSE).

**채점**(각 25점·총 100): PSNR(10→16), accuracy(0.5→0.95), μ-MSE<0.005, σ-MSE<0.005.

`Model.forward(x)` 는 **4-튜플**을 반환: `(denoised[B,1,28,28], label_prob[B,1], mu[B,1], sigma[B,1])`.
이 노트북은 **베이스라인** = 스캐폴드 기본(무작위 출력) → 전 항목 0 → **0점**. 모범답안(다중과제 U-Net) 참고.

**제출**: `submission.npz` — `denoised, label_pred, mu_pred, std_pred` (val 순서, 2000개).


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/train.pkl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-abnormal-distribution/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import pickle, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"

def _img(a):  # (28,28,1) uint8 [0,255] -> (1,28,28) float [0,1]
    return torch.tensor(np.array(a, dtype=np.float32).reshape(28,28)/255.0)[None]

def load_train(f="data/train.pkl"):
    d = pickle.load(open(f, "rb"))
    O = torch.stack([_img(s["original"]) for s in d]); N = torch.stack([_img(s["noised"]) for s in d])
    L = torch.tensor([float(s["label"]) for s in d])
    return O, N, L

def load_val_noised(f="data/val_noised.pkl"):
    return torch.stack([_img(s["noised"]) for s in pickle.load(open(f, "rb"))])   # (2000,1,28,28)

Otr, Ntr, Ltr = load_train(); Nval = load_val_noised()
print("train", Otr.shape[0], "val", Nval.shape[0], "| dev", dev)


In [ ]:
class Model(nn.Module):
    """베이스라인: 스캐폴드 기본 — 무작위 출력(학습 없음). 전 항목 0점."""
    def __init__(self): super().__init__()
    def forward(self, x):
        return (torch.rand_like(x), torch.rand(x.shape[0],1,device=x.device),
                torch.randn(x.shape[0],1,device=x.device), torch.randn(x.shape[0],1,device=x.device))

your_model = Model().to(dev)


In [ ]:
# val 예측 -> submission.npz (denoised, label_pred, mu_pred, std_pred; val 순서)
your_model.eval(); dens=[]; labs=[]; mus=[]; sds=[]
with torch.no_grad():
    for i in range(0, Nval.shape[0], 64):
        den, lp, mp, sp = your_model(Nval[i:i+64].to(dev))
        dens.append(den.cpu().numpy()); labs.append(lp.view(-1).cpu().numpy())
        mus.append(mp.view(-1).cpu().numpy()); sds.append(sp.view(-1).cpu().numpy())
np.savez_compressed("submission.npz",
    denoised=np.concatenate(dens).astype(np.float32),
    label_pred=np.concatenate(labs).astype(np.float32),
    mu_pred=np.concatenate(mus).astype(np.float32),
    std_pred=np.concatenate(sds).astype(np.float32))
print("submission.npz 저장:", Nval.shape[0], "개")


### 다음 단계
무작위 출력은 전 항목 0점. `Model` 을 **다중과제 U-Net**(잡음제거 디코더 + 분류/μ/σ 헤드)으로 만들고,
μ·σ 타깃은 train 의 `noised - original` 픽셀 평균/표준편차로 지도학습하라. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)